# 01 — Exploratory Data Analysis

Every figure below answers a specific ML question about the prepared dataset:

| Figure | Question it answers |
|---|---|
| Class balance per split | Is the label distribution imbalanced? Does it differ between splits? |
| Duration distribution | Is a fixed 4 s feature window appropriate? Any very short/long outliers? |
| Speakers per split | Is the speaker-disjoint split gender-balanced (anti-leakage)? |
| Emotion × intensity | Is there a confound between emotion and recording loudness/intensity? |
| Waveform + log-Mel examples | What do the model inputs look like per emotion? |

> **Prerequisite:** run `make prepare` (or `python scripts/prepare_data.py --config configs/data.yaml`) first so that `data/processed/metadata.csv` exists.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

# Make the package importable whether the kernel starts at the repo root
# or inside notebooks/.
ROOT = Path.cwd()
while not (ROOT / "src" / "emotion_recognition").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

## Dataset at a glance

Loading `metadata.csv` — one row per valid audio-only **speech** utterance with speaker/emotion/intensity/statement labels and the assigned `dataset_split`.

In [ ]:
metadata_path = ROOT / "data" / "processed" / "metadata.csv"
assert metadata_path.is_file(), (
    f"{metadata_path} not found. Run `make prepare` first."
)

metadata = pd.read_csv(metadata_path)
print(f"utterances: {len(metadata):,}")
print(f"speakers:   {metadata['speaker_id'].nunique()}")
print()
print(metadata.head(3).to_string(index=False))
print()
print("utterances by split x emotion:")
print(metadata.groupby(["dataset_split", "emotion"]).size().unstack(fill_value=0))
print()
print("duration (s):")
print(metadata["duration_s"].describe().round(3))
print("\nsampling rates:\n", metadata["sampling_rate"].value_counts().to_string())
print("\nchannels:\n", metadata["channels"].value_counts().to_string())

## Figures

All plotting logic lives in `src/emotion_recognition/utils/figures.py`; the notebook only renders the results.

In [ ]:
from emotion_recognition.utils.figures import generate_eda_figures

figures = generate_eda_figures(metadata, ROOT / "data" / "raw" / "ravdess", ROOT / "reports" / "figures")
display(*[Image(str(p)) for p in figures])

## Interpretation checklist

Fill these in once the real RAVDESS metadata is prepared (values below are example prompts, not results).

- **Class balance:** which emotion has the fewest samples, and why does that match the RAVDESS recording protocol (neutral has no *strong* intensity)?
- **Duration:** min/median/max duration; how much padding does the 4 s window need?
- **Speaker split:** are train/val/test gender-balanced? Confirm no speaker appears in two splits (the pipeline asserts this).
- **Intensity confound:** which emotions are only ever *normal*? What does that imply about classifying them on loudness cues alone?
- **Examples:** do spectrograms of *calm* vs *angry* look qualitatively different?

In [ ]:
# Quick sanity checks every new pipeline run should pass.
assert set(metadata["dataset_split"]) == {"train", "val", "test"}
per_split_speakers = metadata.groupby("dataset_split")["speaker_id"].nunique()
print(per_split_speakers.to_string())
print()
n = metadata["speaker_id"].nunique()
assert sum(per_split_speakers) == n, "a speaker must appear in exactly one split"
print("speaker-disjoint split OK")
print("\nSpeaker->split assignment where a speaker appears in >1 split (should be empty):")
print(metadata.groupby("speaker_id")["dataset_split"].nunique().loc[lambda s: s > 1])